# Transformer from scratch

Manually implement a complete Transformer Engineering block from scratch: Scaled Dot-Product Attention, Multi-Head Attention, and Positional Encodings.

In [1]:
import torch
import torch.nn as nn
import math

## Positional Encodings

In [2]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1), :]

## Scaled Dot-Product Attention

In [3]:
def scaled_dot_product_attention(query, key, value, mask=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    p_attn = torch.softmax(scores, dim=-1)
    return torch.matmul(p_attn, value), p_attn

## Multi-Head Attention

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.d_k = d_model // num_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        q = (
            self.w_q(query)
            .view(batch_size, -1, self.num_heads, self.d_k)
            .transpose(1, 2)
        )
        k = self.w_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        v = (
            self.w_v(value)
            .view(batch_size, -1, self.num_heads, self.d_k)
            .transpose(1, 2)
        )

        x, _ = scaled_dot_product_attention(q, k, v, mask)

        x = (
            x.transpose(1, 2)
            .contiguous()
            .view(batch_size, -1, self.num_heads * self.d_k)
        )
        return self.w_o(x)

## Forward Pass Demonstration

In [5]:
batch_size = 2
seq_length = 10
d_model = 64
num_heads = 8

x = torch.randn(batch_size, seq_length, d_model)

pos_encoder = PositionalEncoding(d_model)
x_pe = pos_encoder(x)

mha = MultiHeadAttention(d_model, num_heads)
output = mha(query=x_pe, key=x_pe, value=x_pe)

print("Input shape: ", x.shape)
print("Output shape:", output.shape)

Input shape:  torch.Size([2, 10, 64])
Output shape: torch.Size([2, 10, 64])
